# Kata 01 — Bucle Agéntico Determinista

> Spec: [`specs/001-agentic-loop/spec.md`](../../specs/001-agentic-loop/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

**Cero mocks.** Todas las celdas de código de este notebook hacen llamadas reales a la API de Anthropic. La forma del bucle es exactamente la que escribirías en producción.

## Setup

Primera celda obligatoria en todos los katas. Si `ANTHROPIC_API_KEY` no está en el entorno, se pide aquí (sin echo). El cliente que devuelve `bootstrap()` está envuelto con un guardia de presupuesto que aborta el notebook antes de quemar la cuenta si entrara en bucle.

In [ ]:
from katas._shared.bootstrap import bootstrap, BudgetExceeded
from katas._shared.eventlog import Logger

client, settings = bootstrap(budget_calls=10)
print("modelo:", settings.model)
print("presupuesto:", settings.budget_calls, "llamadas")

## 1. Concepto en mis palabras

Un agente Claude no termina cuando "dice" que terminó: termina cuando la API devuelve un `stop_reason` terminal. El bucle agéntico es un `while True` cuyo único árbitro es ese campo estructurado.

Tres ramas posibles por turno:

- `stop_reason == "tool_use"` → ejecuto la herramienta indicada, anexo el resultado al historial, sigo iterando.
- `stop_reason == "end_turn"` → corto, devuelvo respuesta final.
- cualquier otro valor (`max_tokens`, ausente, desconocido) → corto **con motivo explícito**; nunca sigo adivinando.

El texto que el modelo genera es **payload**, no señal de control.

## 2. Por qué importa

El primer error de un sistema agéntico en producción es decidir terminación con `if "done" in text`. El día que el modelo:

- parafrasea ("all set" en lugar de "done"),
- responde en otro idioma,
- recibe una entrada del usuario que contiene la frase trampa,

el agente corta antes de tiempo o sigue cuando debía parar. La auditoría no encuentra causa raíz porque el control está oculto en una expresión regular sobre prosa.

El control por `stop_reason` es **determinista** (mismo input → misma transición), **observable** (cada iteración produce una entrada en el log) y **reproducible**.

## 3. Ejemplo correcto (signal-driven)

### 3.1 Declaración de la herramienta

Una sola herramienta `get_weather` con su JSON Schema. El handler local ejecuta lo que el modelo solicite.

In [ ]:
import json

WEATHER_TOOL = {
    "name": "get_weather",
    "description": "Devuelve el clima actual para una ciudad.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "Nombre de la ciudad"}
        },
        "required": ["city"],
    },
}

def get_weather(args: dict) -> dict:
    return {"city": args["city"], "temp_c": 18, "sky": "soleado"}

TOOL_HANDLERS = {"get_weather": get_weather}

### 3.2 El bucle determinista

Lectura clave: **ningún `if` mira el texto del modelo**. Todo el control va por `stop_reason`.

In [ ]:
class UnhandledStop(RuntimeError):
    """stop_reason fuera del conjunto manejado. Nunca silencioso."""

def run_loop_signal(client, *, system, messages, tools, tool_handlers):
    """Bucle controlado por stop_reason. Devuelve (respuesta_final, log)."""
    log = Logger()
    iteration = 0
    while True:
        iteration += 1
        resp = client.messages.create(system=system, messages=messages, tools=tools)
        entry = {"iter": iteration, "stop_reason": resp.stop_reason}

        if resp.stop_reason == "tool_use":
            tu = next(b for b in resp.content if b.type == "tool_use")
            entry["branch"] = "dispatch"
            entry["tool"] = tu.name
            try:
                result = tool_handlers[tu.name](tu.input)
                entry["status"] = "ok"
            except Exception as exc:
                result = {"error": str(exc)}
                entry["status"] = "error"
            messages.append({"role": "assistant", "content": resp.content})
            messages.append({
                "role": "user",
                "content": [{
                    "type": "tool_result",
                    "tool_use_id": tu.id,
                    "content": json.dumps(result),
                }],
            })
            log.add(**entry)
            continue

        if resp.stop_reason == "end_turn":
            entry["branch"] = "halt"
            entry["cause"] = "end_turn"
            log.add(**entry)
            return resp, log

        entry["branch"] = "halt"
        entry["cause"] = f"unhandled:{resp.stop_reason}"
        log.add(**entry)
        raise UnhandledStop(resp.stop_reason)

### 3.3 Caso feliz

Una pregunta sencilla que requiere usar la herramienta. El modelo decide solito invocar `get_weather`; el bucle ejecuta y vuelve a llamar; el modelo cierra con `end_turn`.

In [ ]:
system_basic = "Eres un asistente meteorológico breve. Si necesitas datos, usa la herramienta get_weather."
messages = [{"role": "user", "content": "¿Cómo está el clima en Bogotá?"}]

final, log = run_loop_signal(
    client,
    system=system_basic,
    messages=messages,
    tools=[WEATHER_TOOL],
    tool_handlers=TOOL_HANDLERS,
)

log.show()
print("---")
final_text = next((b.text for b in final.content if b.type == "text"), "")
print("respuesta final:", final_text)
print("usage:", final.usage)

Esperado: dos entradas en el log. La primera con `branch=dispatch` y `tool=get_weather`. La segunda con `branch=halt`, `cause=end_turn`. La traza basta para auditar la corrida.

### 3.4 Caso borde — `stop_reason = max_tokens`

Forzamos truncamiento pidiendo una respuesta larga con un límite ridículo de tokens. El bucle **no** asume "ya terminó porque suena completo": levanta `UnhandledStop` y deja constancia.

In [ ]:
# Forzamos truncamiento bajando max_tokens *sólo* para esta corrida.
saved = settings.max_tokens
settings.max_tokens = 20
try:
    run_loop_signal(
        client,
        system="Responde en español, en mucho detalle, sin abreviar.",
        messages=[{"role": "user", "content": "Explica la historia de la computación en al menos diez párrafos."}],
        tools=[],
        tool_handlers={},
    )
except UnhandledStop as e:
    print("bucle detenido por:", e)
finally:
    settings.max_tokens = saved


## 4. Anti-patrón (lado a lado)

Reemplazo el árbitro `stop_reason` por un buscador de frases en el texto. Es la versión que se ve en muchas demos.

Para que el anti-patrón se dispare de forma reproducible **con la API real**, usamos un *system prompt* que instruye al modelo a anteponer la frase `"task complete"` cuando va a invocar la herramienta. La señal estructurada (`stop_reason=tool_use`) sigue siendo correcta — sólo el texto contiene la trampa. El bucle por prosa caerá; el bucle por señal no.

In [ ]:
DONE_PHRASES = ["task complete", "we are done", "finished", "listo", "done."]

def run_loop_prose(client, *, system, messages, tools, tool_handlers):
    """❌ Anti-patrón: termina cuando ve una frase de cierre en la prosa."""
    log = Logger()
    iteration = 0
    while True:
        iteration += 1
        resp = client.messages.create(system=system, messages=messages, tools=tools)
        text = " ".join(b.text.lower() for b in resp.content if b.type == "text")

        # ❌ Decisión basada en prosa, no en stop_reason
        if any(p in text for p in DONE_PHRASES):
            log.add(iter=iteration, branch="halt_by_prose", text_snippet=text[:80])
            return resp, log

        if resp.stop_reason == "tool_use":
            tu = next(b for b in resp.content if b.type == "tool_use")
            result = tool_handlers[tu.name](tu.input)
            messages.append({"role": "assistant", "content": resp.content})
            messages.append({
                "role": "user",
                "content": [{"type": "tool_result", "tool_use_id": tu.id, "content": json.dumps(result)}],
            })
            log.add(iter=iteration, branch="dispatch", tool=tu.name)
            continue

        log.add(iter=iteration, branch="halt_other", stop_reason=resp.stop_reason)
        return resp, log

In [ ]:
system_trap = (
    "Eres un asistente meteorológico que sigue protocolo. "
    "ANTES de invocar cualquier herramienta, escribe en español la frase exacta "
    "\"task complete\" como confirmación al usuario. Después invoca la herramienta normalmente. "
    "Cuando entregues el resultado final, vuelve a escribir \"task complete\" al final."
)
user_msg = "¿Cómo está el clima en Bogotá?"

print("=== Bucle por prosa (anti-patrón) ===")
_, log_prose = run_loop_prose(
    client,
    system=system_trap,
    messages=[{"role": "user", "content": user_msg}],
    tools=[WEATHER_TOOL],
    tool_handlers=TOOL_HANDLERS,
)
log_prose.show()

print("\n=== Bucle por señal (correcto) ===")
final_signal, log_signal = run_loop_signal(
    client,
    system=system_trap,
    messages=[{"role": "user", "content": user_msg}],
    tools=[WEATHER_TOOL],
    tool_handlers=TOOL_HANDLERS,
)
log_signal.show()
print("\nrespuesta final del bucle correcto:")
print(next((b.text for b in final_signal.content if b.type == "text"), "(sin texto)"))

Observación clave:

- **Bucle por prosa**: termina en la iteración 1 con `halt_by_prose`. **Nunca llamó a `get_weather`**. El usuario no recibe datos.
- **Bucle por señal**: pasa por `dispatch` (la herramienta corrió) y luego `halt(end_turn)` (respuesta completa). Mismo modelo, mismo prompt, mismas frases trampa en el texto: el resultado depende sólo del árbitro del bucle.

Si el modelo ocasionalmente no antepone `task complete` (la API es no determinista), basta correr otra vez la celda — el system prompt es lo bastante directivo para que el caso ocurra la mayoría de las veces.

### 4.1 Aserción defensiva

Si más adelante alguien intenta "optimizar" el bucle correcto reintroduciendo búsqueda de prosa, este chequeo lo detecta sin requerir tests unitarios formales.

In [ ]:
import inspect, re

src = inspect.getsource(run_loop_signal)
smell = re.search(r"\b(re\.search|re\.match|in\s+text|\.find\(|DONE_PHRASES)\b", src)
assert smell is None, f"el bucle correcto no debería buscar prosa: {smell.group()}"
print("OK: run_loop_signal no inspecciona el texto del modelo.")

## 5. Argumento de certificación

- **`stop_reason` es la única señal legítima de control.** Es metadato tipado emitido por la API; no depende de cómo redacte el modelo. Sobrevive a parafraseo, traducción y prompt injection en el texto.
- **Cada `stop_reason` tiene una respuesta predefinida.** `tool_use` → ejecutar y continuar. `end_turn` → cortar. Cualquier otro (`max_tokens`, ausente, valor desconocido) → cortar con motivo explícito y registrar la causa. No hay rama "casi terminó".
- **El log estructurado hace la corrida auditable.** Iteración, señal, rama, herramienta, motivo de corte. Reconstruyo la traza completa sin volver a llamar al modelo.
- **Determinismo Sobre Probabilidad** (Constitución, Principio I): pattern-matching sobre prosa es probabilístico; control por señal es determinista. No hay punto medio defendible.
- **Conexión con otros katas**: el `stop_reason=tool_use` invoca herramientas que en producción atraviesan hooks `PreToolUse` (Kata 02) y `PostToolUse` (Kata 03); los errores estructurados de MCP (Kata 06) llegan como `tool_result` tipado al historial; las escaladas a humano (Kata 16) son otro `tool_use` predefinido.

## 6. Auto-evaluación

**1. ¿Qué ocurre si el modelo devuelve `stop_reason=max_tokens`? ¿Continúo, abortó, reintento?**

Aborto con motivo explícito (`unhandled:max_tokens`), como muestra la celda §3.4. No reintento automático: la respuesta está truncada, así que reintentar sin cambiar nada repetiría el truncamiento. Lo correcto es subir el cap de `max_tokens` para esa llamada, o pedir al modelo que continúe en un turno nuevo (turno explícito iniciado por el cliente, no por el bucle). El bucle deja la decisión al llamador y registra la causa en el log.

**2. Si una herramienta lanza excepción, ¿cómo lo veo en el historial sin romper el invariante "control por señal"?**

Capturo la excepción dentro del bloque `tool_use`, la convierto en un `tool_result` con campo `error`, y la anexo al historial igual que cualquier otro resultado (en `run_loop_signal` lo hago en el bloque `try/except`). El bucle continúa: la próxima iteración mostrará al modelo el error tipado y le permitirá replanificar. El control sigue siendo `stop_reason`; el fallo es payload, no señal.

**3. ¿Qué información mínima debe registrar el log para reconstruir la traza completa sin volver a llamar al modelo?**

Por iteración: índice, `stop_reason`, rama tomada (`dispatch` / `halt`), nombre de la herramienta cuando aplique, status (`ok` / `error`) de la ejecución y, en la iteración terminal, la causa de corte (`end_turn` o `unhandled:<reason>`). Es exactamente lo que produce `Logger.show()`. Para replicar la respuesta del modelo bit-a-bit haría falta también el guion crudo, pero el log de control basta para auditar la lógica del agente.

---

### Apéndice — costo y modelo

Este notebook hizo del orden de 5–7 llamadas al modelo `claude-haiku-4-5-20251001`. El costo total típico está por debajo de USD 0.01. Si querés correr el mismo notebook contra Sonnet, basta:

```python
client, settings = bootstrap(model="claude-sonnet-4-6")
```

El bucle no cambia.